# Sesión 1 — Taller práctico: el informe del trimestre pasado (CONECTA, proyecto A: Retención)

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 1 de 9  
**Tipo:** Notebook práctico

## Instrucciones de entrega

- Equipos de 3 personas. Entreguen este notebook **ejecutado de inicio a fin** como
  `DATAVIZ_sesion_1_proyecto_A_grupo_Y.ipynb` en el plazo acordado en clase.
- Completen las celdas `# TODO` y escriban en las celdas **Su respuesta**.
- Pueden consultar el notebook guiado, pero escriban su propio código. Pueden apoyarse en un asistente de IA para
  concentrarse en lo importante y en las decisiones; declaren al final para qué lo usaron.

## Problemática de negocio

CONECTA S.A. es un operador de telecomunicaciones en Bogotá con dos líneas, **Hogar** (internet fijo) y **Móvil**
(pospago y prepago), con datos sintéticos del curso. La **Gerencia de Retención**
debe llevar al comité directivo de diciembre una propuesta de campaña contra las cancelaciones para el primer
trimestre de 2027: en qué zonas concentrarla y cuánto invertir. El presupuesto es limitado y lo aprueba el comité
con la propuesta. El trimestre pasado la Gerencia presentó un informe de cuatro gráficos y el comité no supo qué
hacer con él. Como analistas, deben:

1. Fijar para quién es el análisis, qué se decide y qué se pregunta.
2. Diagnosticar los cuatro gráficos del informe.
3. Leer el diccionario de datos y clasificar las columnas con su estética apropiada.
4. Rehacer los gráficos defectuosos.
5. Decir qué debería haber concluido el comité y qué hay que revisar antes de confiar en estos datos.

## Preparación del entorno

In [ ]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

In [ ]:
FUENTE_CONECTA = "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026)."
h = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str})
upz = gpd.read_file(DATOS / "upz.geojson")
OFICIALES = set(upz["localidad"])
print(f"{len(h):,} filas cliente-mes y {h.shape[1]} columnas")
display(h.head())

### Los datos: el diccionario de CONECTA

Antes de graficar hay que saber qué es cada columna. El diccionario (`datos/diccionario_conecta.md`) dice qué
representa cada una, su tipo y su unidad. Léanlo completo: lo usarán en la Tarea 3 y en todo el curso. Dos puntos
que conviene tener presentes desde hoy: **cada fila es un cliente en un mes** (no un cliente), e
`ingreso_mensual_cop` es **lo que el cliente paga a CONECTA** (la tarifa o, en prepago, la recarga del mes), no el
ingreso de su hogar.

In [ ]:
from IPython.display import Markdown
display(Markdown((DATOS / "diccionario_conecta.md").read_text(encoding="utf-8")))

## Tarea 1 — Para quién, qué se decide y qué se pregunta

Knaflic (2015, pp. 20–26): antes de graficar, ¿quién es la audiencia y qué necesita saber o hacer? La ficha de hoy no
lleva todavía el indicador: se define en la S2.

**Llenen la ficha del proyecto: destinatario, decisión, restricción (si la hay) y pregunta para el análisis.**

*Consulten en la presentación:* diapositiva 32, «Contexto: quién decide qué» (sección «Contexto: quién decide qué»); diapositiva 33, «La ficha del comité» (sección «Contexto: quién decide qué»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 2 — Diagnosticar el informe del trimestre pasado

Ejecuten la celda sin modificarla: dibuja los cuatro gráficos del informe tal como llegaron al comité.
Clasifiquen cada uno como **feo**, **malo** o **erróneo** (Wilke, 2019, pp. 1–3) y digan por qué. Lean también el
código de la celda: muestra cómo se construyó cada gráfico (qué datos usa y dónde empieza cada eje).

In [ ]:
# Ejecute esta celda sin modificarla y observe el resultado.
activos = h.groupby("mes")["cliente_id"].nunique()
cancelacion = 100 * h.groupby("mes")["cancelo"].mean()
por_localidad = 100 * h[h["localidad"].isin(OFICIALES)].groupby("localidad")["cancelo"].mean().sort_values()
planes = h["plan"].value_counts()
with plt.style.context("default"):
    fig, ejes = plt.subplots(2, 2, figsize=(13, 8.5))
    a = ejes[0, 0]
    a.plot(activos.index, activos.values, color="#FF00FF", linewidth=4, marker="*", markersize=14)
    a.set_facecolor("#FFFF99"); a.grid(True, color="black", linewidth=1.2)
    a.set_title("Clientes activos", fontfamily="serif", fontsize=15, color="red"); a.tick_params(axis="x", rotation=60)
    a = ejes[0, 1]
    a.bar(activos.index, activos.values, color="#4C72B0"); a.tick_params(axis="x", rotation=60)
    a2 = a.twinx(); a2.plot(cancelacion.index, cancelacion.values, color="#DD8452", marker="o", linewidth=3)
    a.set_ylim(activos.min() * 0.97, activos.max() * 1.01); a2.set_ylim(0, cancelacion.max() * 1.05)
    a.set_title("Clientes y cancelación")
    a = ejes[1, 0]
    a.bar(por_localidad.index, por_localidad.values, color="#4C72B0")
    a.set_ylim(2.5, por_localidad.max() * 1.02); a.set_yticks([]); a.tick_params(axis="x", rotation=90, labelsize=8.5)
    a.set_title("Cancelación por localidad")
    a = ejes[1, 1]
    a.pie(planes, labels=planes.index, shadow=True, startangle=90); a.set_title("Clientes por plan")
    fig.tight_layout()
    plt.show()

**Gráfico «Clientes activos»: ¿feo, malo o erróneo? ¿Por qué?**

*Consulten en la presentación:* diapositiva 11, «Tres formas en que una figura falla» (sección «El informe: feo, malo y erróneo»); diapositiva 12, «Gráfico 1 del informe: feo» (sección «El informe: feo, malo y erróneo»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**Gráfico «Clientes y cancelación»: ¿feo, malo o erróneo? ¿Por qué?**

*Consulten en la presentación:* diapositiva 11, «Tres formas en que una figura falla» (sección «El informe: feo, malo y erróneo»); diapositiva 13, «Gráfico 2 del informe: malo» (sección «El informe: feo, malo y erróneo»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**Gráfico «Cancelación por localidad»: ¿feo, malo o erróneo? ¿Por qué?**

*Consulten en la presentación:* diapositiva 11, «Tres formas en que una figura falla» (sección «El informe: feo, malo y erróneo»); diapositiva 16, «Gráfico 3 del informe: erróneo» (sección «El informe: feo, malo y erróneo»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**Gráfico «Clientes por plan»: ¿qué le pasa?**

*Consulten en la presentación:* diapositiva 11, «Tres formas en que una figura falla» (sección «El informe: feo, malo y erróneo»); diapositiva 20, «Gráfico 4 del informe: la torta» (sección «El informe: feo, malo y erróneo»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 3 — El diccionario, los tipos de datos y las estéticas

Con el diccionario de datos a la vista, construyan una tabla `tipos` con una fila por columna (o grupo de columnas
del mismo tipo): tipo de variable según Wilke (2019, p. 9, Tabla 2-1) y la estética apropiada (Wilke, 2019,
pp. 7–8). Atención: el tipo que importa es el de **lo que la columna representa**, no el que pandas le asigna (por
ejemplo, `codigo_upz` es un número, pero es un código).

*Consulten en la presentación:* diapositiva 25, «Datos y estéticas» (sección «Datos, estéticas, escalas y coordenadas»).

In [ ]:
# TODO: construyan la tabla tipos con las columnas Columna, Tipo de variable y Estética apropiada.
# tipos = pd.DataFrame([
#     ("cliente_id", "...", "..."),
#     ("mes, fecha_alta", "...", "..."),
#     # ... una fila por cada columna (o grupo de columnas del mismo tipo) del diccionario
# ], columns=["Columna", "Tipo de variable", "Estética apropiada"])
# display(tipos)
# Su código aquí:

## Tarea 4 — Rehacer los gráficos defectuosos

La celda de la Tarea 2 ya calculó `por_localidad` (cancelación registrada por localidad, en %), `activos` (clientes
activos por mes) y `cancelacion` (cancelación registrada por mes, en %): úsenlos, no hace falta recalcularlos.

1. Rehagan el gráfico erróneo: barras desde cero, ordenadas, con los valores y un título que diga la conclusión.
   El título compara **Kennedy con Chapinero**: cuántas veces es la una la otra con el eje desde cero y cuántas
   parecía con el eje en 2,5 %.
2. Rehagan el gráfico malo: un panel por variable, cada uno con su escala desde cero.

*Consulten en la presentación:* diapositiva 18, «Corrección del gráfico erróneo» (sección «El informe: feo, malo y erróneo»); diapositiva 14, «Corrección del gráfico malo» (sección «El informe: feo, malo y erróneo»).

In [ ]:
# TODO: rehagan el gráfico erróneo (cancelación por localidad) con barras desde cero y un título de conclusión.
# por_localidad ya existe (Tarea 2).
# real = por_localidad["KENNEDY"] / por_localidad["CHAPINERO"]
# aparente = (por_localidad["KENNEDY"] - 2.5) / (por_localidad["CHAPINERO"] - 2.5)
# fig, ax = plt.subplots(figsize=(9, 6.5))
# ax.barh(list(por_localidad.index), por_localidad.values, ...)
# titulo(fig, f"La cancelación en Kennedy es {decimal(real, 1)} veces la de Chapinero, no ...", "...")
# fuente(fig, FUENTE_CONECTA)
# plt.show()
# Su código aquí:

In [ ]:
# TODO: rehagan el gráfico malo con un panel para los clientes activos y otro para la cancelación.
# activos y cancelacion ya existen (Tarea 2).
# fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
# x = range(len(activos))
# a1.plot(x, activos.values, ...); a1.set_ylim(0, ...)
# a2.plot(x, cancelacion.values, ...); a2.set_ylim(0, ...)
# Su código aquí:

**Antes de las preguntas de interpretación.** Ejecuten la celda sin modificarla: resume cada localidad (con su nombre
oficial y los datos todavía sin revisar) con lo que el gráfico corregido no muestra: cuántos clientes tiene, cuántas
cancelaciones registró, en qué puesto queda por número y por tasa, y cuántas UPZ y estratos agrupa.

In [ ]:
# Ejecute esta celda sin modificarla y observe el resultado.
loc = h[h["localidad"].isin(OFICIALES)].groupby("localidad").agg(
    clientes=("cliente_id", "nunique"), cancelaciones=("cancelo", "sum"), tasa_pct=("cancelo", "mean"),
    upz=("codigo_upz", "nunique"), estrato_min=("estrato", "min"), estrato_max=("estrato", "max"))
loc["tasa_pct"] = 100 * loc["tasa_pct"]
loc = loc.sort_values("tasa_pct", ascending=False)
loc["puesto_tasa"] = range(1, len(loc) + 1)
loc["puesto_cancelaciones"] = loc["cancelaciones"].rank(ascending=False, method="min").astype(int)
display(loc.round(2))

## Tarea 5 — Qué debió concluir el comité

La torta de planes ya mostró una pista. Ejecuten la celda sin modificarla: compara los nombres de localidad y de plan
que trae el archivo con los oficiales, y cuenta cuántas filas dejó fuera el informe al quedarse solo con los nombres
oficiales.

In [ ]:
# Ejecute esta celda sin modificarla y observe el resultado.
print(f"Localidades oficiales: {len(OFICIALES)}; nombres distintos de localidad en el archivo: {h['localidad'].nunique()}")
fuera = ~h["localidad"].isin(OFICIALES)
print(f"Filas que el informe dejó fuera por no tener el nombre oficial: {miles(fuera.sum())} de {miles(len(h))}")
print("Ejemplos:", sorted(h.loc[fuera, "localidad"].unique())[:8])
print("Nombres de plan en el archivo:", sorted(h["plan"].unique()))

**¿Qué podía concluir el comité con este informe y qué hay que revisar antes de confiar en estos datos?**

*Consulten en la presentación:* diapositiva 8, «Lo que el informe nos hacía concluir» (sección «El caso y la necesidad»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Preguntas de interpretación

Estas preguntas preparan la S2: sus respuestas son el punto de partida del próximo taller.

**1. ¿Por qué comparar el número de cancelaciones entre localidades no basta para saber dónde se cancela más? Usen la tabla por localidad de la Tarea 4.**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**2. En el gráfico corregido, la Candelaria tiene una tasa parecida a la de localidades mucho más grandes. ¿Confiarían en su tasa tanto como en la de Kennedy? ¿Qué le falta al gráfico para saberlo?**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**3. ¿Por qué la localidad puede no ser la mejor escala para decidir dónde hacer la campaña?**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Interpretación final para el negocio

Para la Gerencia de Retención, en 5–8 viñetas: qué falla en el informe anterior; qué puede y qué no puede concluir el
comité; qué se debe revisar en los datos; cómo será el siguiente informe; uso de IA.

**Su respuesta:** *(Escriba aquí sus viñetas.)*